# SparseSentinel-APT — GUIDE evaluation

Second evaluation of MABTSA on **GUIDE** (Microsoft Security Incident Prediction), whose incident labels (TruePositive / BenignPositive / FalsePositive) were assigned by customer security analysts.

- Dataset: https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction (CDLA-Permissive-2.0)
- Files expected in `C:\Users\LENOVO\Jupyter_Files\SparseSentinel-APT\GUIDE\`
- `ActionGrouped` / `ActionGranular` record post-triage actions and are **never** used as features.

| Cell | Purpose |
|---|---|
| G0 | Environment check |
| G1 | CSV → Parquet conversion and audit |
| G2 | Build incident × aspect stance units, org/incident-disjoint splits |
| G3 | (next) Probes, sparse selection, validation-selected abstention |


## G0 — Environment check

In [ ]:
# Environment check (run first)
import sys, platform
import pandas as pd, numpy as np, pyarrow
print("Python:", sys.version.split()[0], "|", platform.platform())
print("pandas", pd.__version__, "| numpy", np.__version__, "| pyarrow", pyarrow.__version__)


## G1 — Load and audit
First run converts the CSVs to Parquet (10–20 min). Send the printed output back before G2.

In [ ]:
# =============================================================================
# CELL G1: GUIDE (MICROSOFT SECURITY INCIDENT PREDICTION) — LOAD AND AUDIT
# =============================================================================
#
# Dataset: https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction
# License: CDLA-Permissive-2.0
# Labels:  IncidentGrade assigned by customer security analysts
#          (TruePositive / BenignPositive / FalsePositive)
#
# Put GUIDE_Train.csv and GUIDE_Test.csv in GUIDE_DIR, then run this cell.
# It reads the files in chunks (they are several GB), converts them to Parquet
# once, and prints the schema and label/aspect statistics needed to design
# the MABTSA mapping. Send the printed output back before running later cells.
# =============================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

GUIDE_DIR = Path(r"C:\Users\LENOVO\Jupyter_Files\SparseSentinel-APT\GUIDE")
GUIDE_OUT = GUIDE_DIR / "processed"
GUIDE_OUT.mkdir(parents=True, exist_ok=True)

FILES = {
    "train": GUIDE_DIR / "GUIDE_Train.csv",
    "test": GUIDE_DIR / "GUIDE_Test.csv",
}
CHUNK_ROWS = 1_000_000

# Columns that record what analysts DID after triage. They are consequences of
# the label and must never be used as model inputs (label leakage).
POST_TRIAGE_COLUMNS = ["ActionGrouped", "ActionGranular"]

# GUIDE Category values (ATT&CK tactics) mapped to the four DAPT2026-MABTSA
# aspects. Verify against the printed category list below and adjust.
CATEGORY_TO_ASPECT = {
    "Discovery": "reconnaissance",
    "Reconnaissance": "reconnaissance",
    "InitialAccess": "initial_access_foothold",
    "Execution": "initial_access_foothold",
    "Persistence": "initial_access_foothold",
    "CredentialAccess": "initial_access_foothold",
    "LateralMovement": "lateral_movement",
    "Exfiltration": "data_exfiltration",
    "Collection": "data_exfiltration",
}


# -----------------------------------------------------------------------------
# 1. Convert CSV -> Parquet (one-time, chunked, low memory)
# -----------------------------------------------------------------------------

def convert(split, csv_path):
    parquet_path = GUIDE_OUT / f"guide_{split}.parquet"
    if parquet_path.exists():
        return parquet_path
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing {csv_path}")

    import pyarrow as pa
    import pyarrow.parquet as pq

    writer = None
    # Read every column as string so the schema is identical across chunks
    # (GUIDE has sparse columns that are empty in some chunks).
    for chunk in pd.read_csv(csv_path, chunksize=CHUNK_ROWS, dtype=str, keep_default_na=True):
        if writer is None:
            schema = pa.schema([(c, pa.string()) for c in chunk.columns])
            writer = pq.ParquetWriter(parquet_path, schema)
        table = pa.Table.from_pandas(chunk, schema=schema, preserve_index=False)
        writer.write_table(table)
    writer.close()
    return parquet_path


paths = {split: convert(split, path) for split, path in FILES.items()}


# -----------------------------------------------------------------------------
# 2. Audit
# -----------------------------------------------------------------------------

AUDIT_COLUMNS = ["IncidentId", "AlertId", "OrgId", "Timestamp", "Category",
                 "MitreTechniques", "IncidentGrade", "EntityType", "EvidenceRole"]

report = {}
print("=" * 80)
print("GUIDE AUDIT")
print("=" * 80)

for split, path in paths.items():
    import pyarrow.parquet as pq
    columns = pq.read_schema(path).names
    print(f"\n[{split}] {path}")
    print(f"Columns ({len(columns)}): {columns}")

    available = [c for c in AUDIT_COLUMNS if c in columns]
    frame = pd.read_parquet(path, columns=available)

    per_incident = frame.groupby("IncidentId").agg(
        evidence_rows=("AlertId", "size"),
        alerts=("AlertId", "nunique"),
        grades=("IncidentGrade", lambda s: s.dropna().nunique()),
        grade=("IncidentGrade", lambda s: s.dropna().iloc[0] if s.notna().any() else None),
        orgs=("OrgId", "nunique"),
    )

    info = {
        "rows": int(len(frame)),
        "incidents": int(frame["IncidentId"].nunique()),
        "alerts": int(frame["AlertId"].nunique()),
        "orgs": int(frame["OrgId"].nunique()),
        "rows_missing_grade": int(frame["IncidentGrade"].isna().sum()),
        "incidents_with_conflicting_grades": int((per_incident["grades"] > 1).sum()),
        "incidents_spanning_multiple_orgs": int((per_incident["orgs"] > 1).sum()),
        "incident_grade_distribution": per_incident["grade"].value_counts(dropna=False).to_dict(),
        "evidence_rows_per_incident": per_incident["evidence_rows"].describe(
            percentiles=[.5, .75, .9, .95, .99]).round(1).to_dict(),
        "alerts_per_incident": per_incident["alerts"].describe(
            percentiles=[.5, .9, .99]).round(1).to_dict(),
        "category_distribution": frame["Category"].value_counts(dropna=False).to_dict(),
        "entity_types": frame["EntityType"].value_counts().head(40).to_dict() if "EntityType" in frame else None,
        "leakage_columns_present": [c for c in POST_TRIAGE_COLUMNS if c in columns],
    }

    frame["aspect"] = frame["Category"].map(CATEGORY_TO_ASPECT)
    incident_aspects = frame.dropna(subset=["aspect"]).groupby(["IncidentId", "aspect"]).size().reset_index()
    incident_aspects = incident_aspects.merge(per_incident["grade"], left_on="IncidentId", right_index=True)
    info["incident_aspect_by_grade"] = (
        incident_aspects.groupby(["aspect", "grade"]).size().unstack(fill_value=0).to_dict()
    )
    info["unmapped_categories"] = sorted(
        set(frame["Category"].dropna().unique()) - set(CATEGORY_TO_ASPECT)
    )
    report[split] = info

    for key, value in info.items():
        print(f"\n{key}:")
        if isinstance(value, dict):
            print(pd.Series(value, dtype=object).to_string() if value and not isinstance(next(iter(value.values())), dict)
                  else pd.DataFrame(value).to_string())
        else:
            print(value)

with open(GUIDE_OUT / "guide_audit.json", "w", encoding="utf-8") as file:
    json.dump(report, file, indent=2, default=str)

print("\nSaved audit to", GUIDE_OUT / "guide_audit.json")
print("Send this printed output (or guide_audit.json) back to design the stance mapping.")


## G2 — Build MABTSA stance units and evidence windows
Analyst grade → stance; key = (OrgId, IncidentId); org-disjoint train/validation; official test. Send the printed output back before G3.

In [ ]:
# =============================================================================
# CELL G2: GUIDE -> MABTSA STANCE UNITS AND EVIDENCE WINDOWS
# =============================================================================
#
# Builds (incident, aspect) stance units with ANALYST-ASSIGNED labels and a
# fixed-length evidence window per incident.
#
# Key decisions (all recorded in the manifest):
#   * Incident key = (OrgId, IncidentId). IncidentId is NOT globally unique.
#   * Incidents with missing or conflicting analyst grades are dropped.
#   * Stance mapping (follows the DAPT2026-MABTSA annotation guideline):
#       TruePositive  + aspect present            -> SUPPORTING
#       FalsePositive + aspect present            -> CONTRADICTING
#       BenignPositive+ aspect present            -> CONTRADICTING (BP_POLICY)
#       TruePositive  + aspect absent             -> NEUTRAL
#           ("evidence concerns a different attack aspect")
#       FP/BP         + aspect absent             -> no unit
#           ("absence of attack evidence is not contradiction")
#   * Splits: GUIDE_Train -> organisation-disjoint train/validation;
#             GUIDE_Test  -> official test (org overlap with train reported).
#   * ActionGrouped / ActionGranular (post-triage actions) and Usage are
#     never read.
# =============================================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

GUIDE_DIR = Path(r"C:\Users\LENOVO\Jupyter_Files\SparseSentinel-APT\GUIDE")
PROCESSED = GUIDE_DIR / "processed"
OUT = PROCESSED / "mabtsa"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 20261013
WINDOW = 32
VALIDATION_ORG_FRACTION = 0.15

# "contradicting": BenignPositive is a benign interpretation -> CONTRADICTING
# "exclude":       drop BenignPositive incidents (report as sensitivity analysis)
BP_POLICY = "contradicting"

# NEUTRAL units sampled per TruePositive incident (from its absent aspects).
NEUTRAL_PER_TP_INCIDENT = 1

# Unit caps per split (None = keep all). Train is stratified by aspect x stance;
# validation/test are uniform samples that keep the natural distribution.
MAX_UNITS = {"train": 80_000, "validation": 20_000, "test": None}

CATEGORY_TO_ASPECT = {
    "Discovery": "reconnaissance",
    "InitialAccess": "initial_access_foothold",
    "Execution": "initial_access_foothold",
    "Persistence": "initial_access_foothold",
    "CredentialAccess": "initial_access_foothold",
    "LateralMovement": "lateral_movement",
    "Exfiltration": "data_exfiltration",
    "Collection": "data_exfiltration",
}
ASPECTS = ["reconnaissance", "initial_access_foothold",
           "lateral_movement", "data_exfiltration"]

GRADE_TO_STANCE = {
    "TruePositive": "SUPPORTING",
    "FalsePositive": "CONTRADICTING",
    "BenignPositive": "CONTRADICTING",
}

CATEGORICAL = ["Category", "EntityType", "EvidenceRole", "DetectorId", "AlertTitle",
               "SuspicionLevel", "LastVerdict", "OSFamily", "ResourceType", "Roles",
               "AntispamDirection", "ThreatFamily", "CountryCode"]
MAX_TECHNIQUES_PER_ROW = 4
VOCAB_MIN_COUNT = 5
VOCAB_MAX_SIZE = 50_000

READ_COLUMNS = ["OrgId", "IncidentId", "AlertId", "Timestamp", "MitreTechniques",
                "IncidentGrade"] + CATEGORICAL
FORBIDDEN = {"ActionGrouped", "ActionGranular", "Usage"}
assert FORBIDDEN.isdisjoint(READ_COLUMNS)


def h(value, salt=""):
    return int(hashlib.sha256(f"{SEED}|{salt}|{value}".encode()).hexdigest()[:15], 16)


def read(split_file, columns):
    path = PROCESSED / f"guide_{split_file}.parquet"
    available = set(pq.read_schema(path).names)
    columns = [c for c in columns if c in available]
    table = pq.read_table(path, columns=columns, read_dictionary=columns)
    frame = table.to_pandas()
    # Prefix with the source file: the same (OrgId, IncidentId) can occur in both files.
    frame["key"] = (split_file + ":" + frame["OrgId"].astype(str) + "_"
                    + frame["IncidentId"].astype(str))
    return frame


# -----------------------------------------------------------------------------
# 1. Incident-level labels and aspects
# -----------------------------------------------------------------------------

audit = {}
incident_tables = {}

for split_file in ["train", "test"]:
    rows = read(split_file, ["OrgId", "IncidentId", "Category", "IncidentGrade"])
    rows["aspect"] = rows["Category"].astype(str).map(CATEGORY_TO_ASPECT)

    grades = rows.dropna(subset=["IncidentGrade"]).groupby("key", observed=True)["IncidentGrade"]
    grade_counts = grades.nunique()
    incident = pd.DataFrame({"grade": grades.first().astype(str), "n_grades": grade_counts})
    incident["OrgId"] = rows.groupby("key", observed=True)["OrgId"].first().astype(str)

    all_keys = rows["key"].nunique()
    conflicting = int((incident["n_grades"] > 1).sum())
    incident = incident[incident["n_grades"] == 1].drop(columns="n_grades")

    present = (rows.dropna(subset=["aspect"])
               .groupby("key", observed=True)["aspect"].agg(lambda s: sorted(set(s))))
    incident["aspects"] = present.reindex(incident.index)
    incident["aspects"] = incident["aspects"].apply(lambda v: v if isinstance(v, list) else [])

    audit[split_file] = {
        "incidents_by_org_incident_key": int(all_keys),
        "dropped_missing_grade": int(all_keys - grade_counts.shape[0]),
        "dropped_conflicting_grade": conflicting,
        "kept_incidents": int(len(incident)),
        "kept_grade_distribution": incident["grade"].value_counts().to_dict(),
        "kept_with_any_mapped_aspect": int((incident["aspects"].str.len() > 0).sum()),
    }
    incident_tables[split_file] = incident
    del rows


# -----------------------------------------------------------------------------
# 2. Splits (organisation-disjoint train/validation; official test)
# -----------------------------------------------------------------------------

bare = {f: set(t.index.str.split(":", n=1).str[1]) for f, t in incident_tables.items()}
audit["org_incident_keys_in_both_files"] = len(bare["train"] & bare["test"])

train_incidents = incident_tables["train"]
orgs = pd.Series(train_incidents["OrgId"].unique())
validation_orgs = set(orgs[orgs.map(lambda o: h(o, "valorg") % 10_000 < VALIDATION_ORG_FRACTION * 10_000)])
train_incidents["split"] = np.where(train_incidents["OrgId"].isin(validation_orgs), "validation", "train")

test_incidents = incident_tables["test"]
test_incidents["split"] = "test"
train_orgs = set(train_incidents.loc[train_incidents["split"] == "train", "OrgId"])
test_incidents["org_seen_in_train"] = test_incidents["OrgId"].isin(train_orgs)
train_incidents["org_seen_in_train"] = train_incidents["split"] == "train"

incidents = pd.concat([train_incidents, test_incidents])
incidents["source_file"] = np.where(incidents["split"] == "test", "test", "train")


# -----------------------------------------------------------------------------
# 3. Stance units
# -----------------------------------------------------------------------------

unit_rows = []
for key, inc in incidents.iterrows():
    grade, present = inc["grade"], inc["aspects"]
    if grade == "BenignPositive" and BP_POLICY == "exclude":
        continue
    for aspect in present:
        unit_rows.append((key, aspect, GRADE_TO_STANCE[grade], grade, "aspect_present"))
    if grade == "TruePositive" and NEUTRAL_PER_TP_INCIDENT > 0:
        absent = [a for a in ASPECTS if a not in present]
        absent.sort(key=lambda a: h(f"{key}|{a}", "neutral"))
        for aspect in absent[:NEUTRAL_PER_TP_INCIDENT]:
            unit_rows.append((key, aspect, "NEUTRAL", grade, "aspect_absent"))

units = pd.DataFrame(unit_rows, columns=["key", "aspect", "stance", "analyst_grade", "unit_origin"])
units = units.merge(incidents[["split", "OrgId", "org_seen_in_train", "source_file"]],
                    left_on="key", right_index=True, how="left")
units["rank"] = [h(f"{k}|{a}", "sample") for k, a in zip(units["key"], units["aspect"])]

before = units.groupby(["split", "aspect", "stance"]).size()
audit["units_before_sampling"] = {"|".join(k): int(v) for k, v in before.items()}


def cap(frame, limit, stratified):
    if limit is None or len(frame) <= limit:
        return frame
    if not stratified:
        return frame.nsmallest(limit, "rank")
    groups = {k: g.sort_values("rank") for k, g in frame.groupby(["aspect", "stance"])}
    per = {k: 0 for k in groups}
    remaining, chosen = limit, []
    # round-robin quota so rare strata are kept in full
    while remaining > 0:
        open_groups = [k for k in groups if per[k] < len(groups[k])]
        if not open_groups:
            break
        share = max(1, remaining // len(open_groups))
        for k in open_groups:
            take = min(share, len(groups[k]) - per[k], remaining)
            chosen.append(groups[k].iloc[per[k]:per[k] + take])
            per[k] += take
            remaining -= take
            if remaining == 0:
                break
    return pd.concat(chosen)


units = pd.concat([
    cap(units[units["split"] == s], MAX_UNITS[s], stratified=(s == "train"))
    for s in ["train", "validation", "test"]
]).drop(columns="rank").reset_index(drop=True)
units["unit_id"] = ["GUIDE-U-" + format(h(f"{k}|{a}", "id"), "015X")
                    for k, a in zip(units["key"], units["aspect"])]
assert units["unit_id"].is_unique


# -----------------------------------------------------------------------------
# 4. Evidence windows for every selected incident
# -----------------------------------------------------------------------------

selected = units[["key", "source_file"]].drop_duplicates("key")
window_parts = []
for source_file, keys in selected.groupby("source_file")["key"]:
    rows = read(source_file, READ_COLUMNS)
    rows = rows[rows["key"].isin(set(keys))].copy()
    rows["ts"] = pd.to_datetime(rows["Timestamp"].astype(str), errors="coerce", utc=True)
    rows = rows.sort_values(["key", "ts", "AlertId"], kind="stable")
    rows["position"] = rows.groupby("key", observed=True).cumcount()
    sizes = rows.groupby("key", observed=True).size()
    rows = rows[rows["position"] < WINDOW]
    rows["incident_rows_total"] = rows["key"].map(sizes)
    window_parts.append(rows)
    del rows
evidence = pd.concat(window_parts, ignore_index=True)
for column in CATEGORICAL + ["MitreTechniques", "AlertId"]:
    if column in evidence:
        evidence[column] = (evidence[column].astype(object).where(evidence[column].notna(), "")
                            .astype(str).replace({"nan": "", "None": "", "<NA>": ""}))

incident_keys = selected["key"].tolist()
incident_index = {k: i for i, k in enumerate(incident_keys)}
evidence["incident_index"] = evidence["key"].map(incident_index).astype(np.int64)
units["incident_index"] = units["key"].map(incident_index).astype(np.int64)

# Vocabularies are fitted on TRAIN-split evidence only.
train_keys = set(units.loc[units["split"] == "train", "key"])
is_train_row = evidence["key"].isin(train_keys)
vocab = {}
for column in [c for c in CATEGORICAL if c in evidence]:
    counts = evidence.loc[is_train_row, column].value_counts()
    counts = counts[(counts >= VOCAB_MIN_COUNT) & (counts.index != "")].head(VOCAB_MAX_SIZE)
    vocab[column] = {v: i + 2 for i, v in enumerate(counts.index)}  # 0 pad, 1 unknown/empty

tech_lists = evidence["MitreTechniques"].str.split(";").apply(lambda l: [t.strip() for t in l if t.strip()])
tech_counts = pd.Series([t for l in tech_lists[is_train_row] for t in l]).value_counts()
tech_vocab = {t: i + 2 for i, t in enumerate(tech_counts[tech_counts >= VOCAB_MIN_COUNT].index)}

n_inc, n_cat = len(incident_keys), len(vocab)
cat_ids = np.zeros((n_inc, WINDOW, n_cat), dtype=np.int32)
tech_ids = np.zeros((n_inc, WINDOW, MAX_TECHNIQUES_PER_ROW), dtype=np.int32)
temporal = np.zeros((n_inc, WINDOW, 3), dtype=np.float32)
mask = np.zeros((n_inc, WINDOW), dtype=bool)

ii = evidence["incident_index"].to_numpy()
pp = evidence["position"].to_numpy()
mask[ii, pp] = True
for j, column in enumerate(vocab):
    cat_ids[ii, pp, j] = evidence[column].map(vocab[column]).fillna(1).astype(np.int32).to_numpy()
for r, (i, p, l) in enumerate(zip(ii, pp, tech_lists)):
    ids = [tech_vocab.get(t, 1) for t in l][:MAX_TECHNIQUES_PER_ROW]
    tech_ids[i, p, :len(ids)] = ids

ts = evidence["ts"]
start = ts.groupby(evidence["key"]).transform("min")
previous = ts.groupby(evidence["key"]).shift(1)
temporal[ii, pp, 0] = np.log1p((ts - start).dt.total_seconds().fillna(0).clip(lower=0)).to_numpy()
temporal[ii, pp, 1] = np.log1p((ts - previous).dt.total_seconds().fillna(0).clip(lower=0)).to_numpy()
temporal[ii, pp, 2] = (ts.dt.hour.fillna(0) / 23.0).to_numpy()

# Deterministic verbalisation for the text view (anonymised IDs stay as tokens).
evidence["evidence_text"] = (
    evidence["Category"] + " alert"
    + np.where(evidence["MitreTechniques"] != "", " (" + evidence["MitreTechniques"] + ")", "")
    + " on " + evidence["EntityType"] + " entity as " + evidence["EvidenceRole"]
    + "; suspicion " + evidence["SuspicionLevel"].replace("", "none")
    + "; last verdict " + evidence["LastVerdict"].replace("", "none")
    + "; detector " + evidence["DetectorId"] + "; title " + evidence["AlertTitle"] + "."
)


# -----------------------------------------------------------------------------
# 5. Save
# -----------------------------------------------------------------------------

units.to_parquet(OUT / "guide_mabtsa_units.parquet", index=False)
evidence[["incident_index", "key", "position", "AlertId", "Timestamp", "evidence_text",
          "incident_rows_total"]].to_parquet(OUT / "guide_mabtsa_evidence_text.parquet", index=False)
np.savez_compressed(OUT / "guide_mabtsa_windows.npz", cat_ids=cat_ids, tech_ids=tech_ids,
                    temporal=temporal, mask=mask, incident_keys=np.array(incident_keys))
with open(OUT / "guide_mabtsa_vocab.json", "w", encoding="utf-8") as f:
    json.dump({"categorical": vocab, "techniques": tech_vocab,
               "categorical_order": list(vocab)}, f)

truncated = evidence.drop_duplicates("key")["incident_rows_total"].gt(WINDOW).mean()
manifest = {
    "source": "GUIDE (Microsoft Security Incident Prediction), CDLA-Permissive-2.0",
    "label_source": "IncidentGrade assigned by customer security analysts",
    "incident_key": "(OrgId, IncidentId)",
    "stance_mapping": {"TruePositive+present": "SUPPORTING",
                       "FalsePositive+present": "CONTRADICTING",
                       "BenignPositive+present": "CONTRADICTING" if BP_POLICY == "contradicting" else "excluded",
                       "TruePositive+absent": f"NEUTRAL ({NEUTRAL_PER_TP_INCIDENT} sampled per incident)",
                       "FP/BP+absent": "no unit"},
    "category_to_aspect": CATEGORY_TO_ASPECT,
    "splits": "GUIDE_Train -> organisation-disjoint train/validation; GUIDE_Test -> official test",
    "max_units": MAX_UNITS, "window": WINDOW, "seed": SEED,
    "excluded_columns": sorted(FORBIDDEN),
    "incident_windows": n_inc, "share_incidents_truncated_to_window": float(truncated),
    "audit": audit,
}
with open(OUT / "guide_mabtsa_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)


# -----------------------------------------------------------------------------
# Report
# -----------------------------------------------------------------------------

print("=" * 80)
print("GUIDE -> MABTSA UNITS")
print("=" * 80)
for split_file, info in audit.items():
    if split_file in ("train", "test"):
        print(f"\n[{split_file}] incident cleaning:")
        for k, v in info.items():
            print(f"  {k}: {v}")

print("\nUnits per split and stance:")
print(pd.crosstab(units["split"], units["stance"], margins=True).to_string())
print("\nUnits per split, aspect and stance:")
print(pd.crosstab([units["split"], units["aspect"]], units["stance"]).to_string())
print("\nMajority-class share per split (the accuracy floor every model must beat):")
print(units.groupby("split")["stance"].agg(lambda s: s.value_counts(normalize=True).max()).round(4).to_string())
print("(OrgId, IncidentId) keys present in both GUIDE files:",
      audit["org_incident_keys_in_both_files"])
print("Test units from organisations seen in train:",
      f"{units.loc[units['split'] == 'test', 'org_seen_in_train'].mean():.1%}")
print(f"\nIncident windows: {n_inc:,}   truncated to {WINDOW} rows: {truncated:.1%}")
print("Categorical vocab sizes:", {k: len(v) + 2 for k, v in vocab.items()},
      "| techniques:", len(tech_vocab) + 2)
print("\nExample evidence text:")
print(evidence["evidence_text"].head(3).to_string())
print("\nSaved to", OUT)
